<a href="https://colab.research.google.com/github/sabahoth01/NLP-courses-SPBU/blob/task3/task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install -q bitsandbytes
# !pip install -q git+https://github.com/huggingface/peft.git
# !pip install -q transformers datasets accelerate
# !pip install -q gradio
# !pip install -U datasets


In [ ]:
# Загружаем модель GPT-2
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch

In [ ]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Adding pad token to avoid padding errors

model = AutoModelForCausalLM.from_pretrained(model_name)

# Load the wikitext-103 dataset
dataset = load_dataset("wikitext", "wikitext-103-raw-v1")
train_data = dataset['train'].shuffle(seed=42).select(range(10000))  # Limit to 10k examples

# Preprocessing the data
def preprocess(examples):
    prompt = [f"Write the continuation for: {text}" for text in examples['text']]
    tokenized = tokenizer(prompt, truncation=True, padding="max_length", max_length=512)
    labels = tokenizer(examples['text'], truncation=True, padding="max_length", max_length=512)
    return {'input_ids': tokenized['input_ids'], 'labels': labels['input_ids']}

tokenized_data = train_data.map(preprocess, batched=True, remove_columns=train_data.column_names)

# Set up training arguments
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    save_steps=100,
    save_total_limit=2,
    logging_dir='./logs',
    learning_rate=5e-5,
    fp16=False,  # Disable fp16 for CPU
    report_to='none'  # Disable W&B
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
    data_collator=data_collator
)

trainer.train()

# Save the fine-tuned model
trainer.save_model("./gpt2-finetuned")


train-00000-of-00002.parquet:   0%|          | 0.00/157M [00:00<?, ?B/s]

train-00001-of-00002.parquet:   0%|          | 0.00/157M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,3.220800
1000,3.112300


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aa4f6682d70c37d61e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Gradio demo
import gradio as gr


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


def generate_story(prompt, tone="funny"):
    input_prompt = f"Write a {tone} article or continuation for: {prompt}"
    inputs = tokenizer(input_prompt, return_tensors="pt").to(device)  # Move inputs to the same device as model
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

demo = gr.Interface(
    fn=generate_story,
    inputs=[gr.Textbox(label="Story beginning"), gr.Radio(["funny", "serious", "scientific"], label="Tone")],
    outputs="text",
    title="Story Generator (wikitext-103)"
)
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://56d1a31208928ba367.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
